# Kyrgyzstan Education Statistics — Pure Statistical Analysis

**No visualisations. All results expressed as numerical statistics, tests, and tables.**

Datasets: `literacy.csv`, `students_cis.csv`, `students_non_cis.csv`, `education_budget.csv`

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import linregress, shapiro, mannwhitneyu, spearmanr
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 30)
print('Libraries loaded.')

In [ ]:
literacy_raw  = pd.read_csv('literacy.csv',        encoding='utf-8-sig')
cis_raw       = pd.read_csv('students_cis.csv',     encoding='utf-8-sig')
non_cis_raw   = pd.read_csv('students_non_cis.csv', encoding='utf-8-sig')
budget_raw    = pd.read_csv('education_budget.csv', encoding='utf-8-sig')
print('Files loaded. Shapes:', literacy_raw.shape, cis_raw.shape, non_cis_raw.shape, budget_raw.shape)

## 2. Data Cleaning

In [ ]:
# ── Literacy ──────────────────────────────────────────────────────────────────
literacy = literacy_raw[['Items','1989','1999','2009','2022']].copy()
literacy.columns = ['Category','1989','1999','2009','2022']
literacy['Category'] = literacy['Category'].str.strip()
for col in ['1989','1999','2009','2022']:
    literacy[col] = pd.to_numeric(literacy[col], errors='coerce')
literacy_clean = literacy[literacy['Category'].notna() & (literacy['Category']!='') & literacy['1989'].notna()].copy()

# ── International students ────────────────────────────────────────────────────
year_cols = [str(y) for y in range(2010, 2025)]

def extract_by_country(df, group_label):
    en_col = df.columns[2]
    skip = {'total','ncluding:','including:','','nan'}
    rows = []
    for _, row in df.iterrows():
        name = str(row[en_col]).strip()
        if name.lower() in skip or 'Including' in name:
            continue
        vals = pd.to_numeric(pd.Series([row[y] for y in year_cols]), errors='coerce').values
        rows.append({'Country': name, 'Group': group_label, **dict(zip(year_cols, vals))})
    return pd.DataFrame(rows)

def extract_totals(df):
    en_col = df.columns[2]
    mask = df[en_col].astype(str).str.strip().str.lower().isin(['total'])
    row = df[mask].iloc[0]
    return pd.to_numeric(row[year_cols], errors='coerce')

cis_countries     = extract_by_country(cis_raw, 'CIS')
non_cis_countries = extract_by_country(non_cis_raw, 'Non-CIS')
countries_df      = pd.concat([cis_countries, non_cis_countries], ignore_index=True)

cis_total     = extract_totals(cis_raw)
non_cis_total = extract_totals(non_cis_raw)
students_df   = pd.DataFrame({'Year':[int(y) for y in year_cols],
                              'CIS':cis_total.values, 'Non_CIS':non_cis_total.values})
students_df['Total']      = students_df['CIS'] + students_df['Non_CIS']
students_df['CIS_pct']    = students_df['CIS']     / students_df['Total'] * 100
students_df['NonCIS_pct'] = students_df['Non_CIS'] / students_df['Total'] * 100

# ── Budget ────────────────────────────────────────────────────────────────────
budget = budget_raw.copy()
en_col_b = budget.columns[2]
budget[en_col_b] = budget[en_col_b].astype(str).str.strip()
b_years = [str(y) for y in range(2001, 2025)]
for col in b_years:
    if col in budget.columns:
        budget[col] = pd.to_numeric(budget[col], errors='coerce')

def get_budget_row(kw):
    mask = budget[en_col_b].str.lower().str.contains(kw, na=False)
    return budget[mask].iloc[0][b_years].astype(float) if mask.any() else None

budget_ts = pd.DataFrame({'Year':[int(y) for y in b_years],
    'Total_mln_som': get_budget_row('total').values,
    'Preschool':     get_budget_row('pre-school').values if get_budget_row('pre-school') is not None else np.nan,
    'Higher':        get_budget_row('higher').values if get_budget_row('higher') is not None else np.nan,
})
print('Data cleaned. students_df:', students_df.shape, '| budget_ts:', budget_ts.shape)

## 3. Descriptive Statistics

In [ ]:
print('=== 3.1 International Students — Descriptive Statistics ===')
display(students_df[['CIS','Non_CIS','Total']].describe().round(1))

print('\n=== 3.2 Year-on-Year Growth Rates (%) ===')
growth = students_df[['Year','CIS','Non_CIS','Total']].copy()
growth['CIS_growth%']     = growth['CIS'].pct_change() * 100
growth['NonCIS_growth%']  = growth['Non_CIS'].pct_change() * 100
growth['Total_growth%']   = growth['Total'].pct_change() * 100
display(growth.round(2))

print('\n=== 3.3 Students by Country — Total 2010-2024 ===')
country_totals = countries_df.copy()
country_totals['Total_2010_2024'] = country_totals[year_cols].sum(axis=1)
country_totals['Mean_per_year']   = country_totals[year_cols].mean(axis=1).round(1)
country_totals['Max_year_value']  = country_totals[year_cols].max(axis=1)
country_totals['Min_year_value']  = country_totals[year_cols].min(axis=1)
display(country_totals[['Country','Group','Total_2010_2024','Mean_per_year','Max_year_value','Min_year_value']].sort_values('Total_2010_2024', ascending=False).reset_index(drop=True))

In [ ]:
print('=== 3.4 Education Budget — Descriptive Statistics (mln som) ===')
display(budget_ts[['Total_mln_som','Preschool','Higher']].describe().round(1))

print('\n=== 3.5 Budget Growth Summary ===')
b_start = budget_ts['Total_mln_som'].iloc[0]
b_end   = budget_ts['Total_mln_som'].iloc[-1]
b_cagr  = (b_end / b_start) ** (1 / (len(budget_ts)-1)) - 1
print(f'  Budget 2001:   {b_start:,.0f} mln som')
print(f'  Budget 2024:   {b_end:,.0f} mln som')
print(f'  Nominal growth: {(b_end/b_start - 1)*100:.1f}%')
print(f'  CAGR (2001-2024): {b_cagr*100:.2f}% per year')

print('\n=== 3.6 Student Enrollment Summary ===')
for grp, col in [('CIS','CIS'),('Non-CIS','Non_CIS'),('Total','Total')]:
    s = students_df[col]
    cagr = (s.iloc[-1]/s.iloc[0])**(1/(len(s)-1))-1
    print(f'  {grp:8s} | 2010: {s.iloc[0]:>7,.0f} | 2024: {s.iloc[-1]:>7,.0f} | Peak: {s.max():>7,.0f} ({students_df.loc[s.idxmax(),"Year"]}) | CAGR: {cagr*100:.2f}%')

## 4. Normality Tests (Shapiro-Wilk)

In [ ]:
print('=== 4. Shapiro-Wilk Normality Tests (H0: data is normally distributed) ===')
series_to_test = {
    'CIS students':            students_df['CIS'],
    'Non-CIS students':        students_df['Non_CIS'],
    'Total students':          students_df['Total'],
    'Education budget':        budget_ts['Total_mln_som'],
    'CIS YoY growth%':         students_df['CIS'].pct_change().dropna()*100,
    'Non-CIS YoY growth%':     students_df['Non_CIS'].pct_change().dropna()*100,
}
rows = []
for name, s in series_to_test.items():
    w, p = shapiro(s.dropna())
    rows.append({'Series': name, 'W-statistic': round(w,4), 'p-value': round(p,4),
                 'Normal (α=0.05)': 'Yes' if p > 0.05 else 'No'})
display(pd.DataFrame(rows))

## 5. Correlation Analysis

In [ ]:
merged = students_df.merge(budget_ts[budget_ts['Year']>=2010][['Year','Total_mln_som']], on='Year')

print('=== 5.1 Pearson Correlations ===')
pairs = [
    ('Total students', 'Education budget', 'Total', 'Total_mln_som'),
    ('CIS students',   'Education budget', 'CIS',   'Total_mln_som'),
    ('Non-CIS students','Education budget','Non_CIS','Total_mln_som'),
    ('CIS students',   'Non-CIS students', 'CIS',   'Non_CIS'),
]
for lx, ly, cx, cy in pairs:
    r, p = stats.pearsonr(merged[cx], merged[cy])
    print(f'  {lx:25s} vs {ly:25s} | r = {r:+.4f} | p = {p:.4f} | {"Significant" if p<0.05 else "Not significant"} (α=0.05)')

print('\n=== 5.2 Spearman Correlations (rank-based) ===')
for lx, ly, cx, cy in pairs:
    r, p = spearmanr(merged[cx], merged[cy])
    print(f'  {lx:25s} vs {ly:25s} | ρ = {r:+.4f} | p = {p:.4f} | {"Significant" if p<0.05 else "Not significant"} (α=0.05)')

print('\n=== 5.3 Full Correlation Matrix ===')
corr_df = merged[['CIS','Non_CIS','Total','Total_mln_som']].copy()
corr_df.columns = ['CIS Students','Non-CIS Students','Total Students','Education Budget']
display(corr_df.corr().round(4))

## 6. Linear Regression: Budget → Total Students

In [ ]:
from scipy.stats import linregress

X = merged['Total_mln_som'].values
Y = merged['Total'].values

slope, intercept, r_value, p_value, std_err = linregress(X, Y)
n = len(X)
r2 = r_value**2
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - 2)

# Residuals
Y_pred   = slope * X + intercept
residuals = Y - Y_pred
rmse      = np.sqrt(np.mean(residuals**2))
mae       = np.mean(np.abs(residuals))

# 95% CI for slope
t_crit = stats.t.ppf(0.975, df=n-2)
ci_low  = slope - t_crit * std_err
ci_high = slope + t_crit * std_err

print('=== 6. Simple Linear Regression: Total Students ~ Education Budget ===')
print(f'  n               = {n}')
print(f'  Slope           = {slope:.5f}  (students per 1 mln som)')
print(f'  95% CI slope    = [{ci_low:.5f}, {ci_high:.5f}]')
print(f'  Intercept       = {intercept:,.2f}')
print(f'  R               = {r_value:.4f}')
print(f'  R²              = {r2:.4f}')
print(f'  Adjusted R²     = {adj_r2:.4f}')
print(f'  p-value         = {p_value:.6f}')
print(f'  Std Error       = {std_err:.5f}')
print(f'  RMSE            = {rmse:,.0f}')
print(f'  MAE             = {mae:,.0f}')

print('\n=== 6.1 Residuals per Year ===')
resid_df = merged[['Year','Total','Total_mln_som']].copy()
resid_df['Predicted'] = Y_pred.round(0)
resid_df['Residual']  = residuals.round(0)
resid_df['Resid%']    = (residuals / Y * 100).round(2)
display(resid_df)

## 7. Hypothesis Test — CIS vs Non-CIS Growth Rates

In [ ]:
cis_growth_rates     = students_df['CIS'].pct_change().dropna().replace([np.inf,-np.inf], np.nan).dropna() * 100
non_cis_growth_rates = students_df['Non_CIS'].pct_change().dropna().replace([np.inf,-np.inf], np.nan).dropna() * 100

print('=== 7.1 Descriptive Statistics of Annual Growth Rates ===')
for name, s in [('CIS', cis_growth_rates), ('Non-CIS', non_cis_growth_rates)]:
    print(f'  {name:8s} | n={len(s)} | Mean={s.mean():.2f}% | Median={s.median():.2f}% | Std={s.std():.2f}% | Min={s.min():.2f}% | Max={s.max():.2f}%')

print('\n=== 7.2 Welch t-test (unequal variances) — one-tailed ===')
print('  H0: mean growth Non-CIS ≤ mean growth CIS')
print('  H1: mean growth Non-CIS >  mean growth CIS')
t_stat, p_two = stats.ttest_ind(non_cis_growth_rates, cis_growth_rates, equal_var=False)
p_one = p_two / 2
print(f'  t-statistic = {t_stat:.4f}')
print(f'  p (two-tailed) = {p_two:.4f}')
print(f'  p (one-tailed) = {p_one:.4f}')
print(f'  Decision (α=0.05): {"Reject H0 — Non-CIS grew significantly faster" if p_one < 0.05 else "Fail to reject H0"}')

print('\n=== 7.3 Mann-Whitney U test (non-parametric alternative) ===')
u_stat, p_mw = mannwhitneyu(non_cis_growth_rates, cis_growth_rates, alternative='greater')
print(f'  U-statistic = {u_stat:.1f}')
print(f'  p-value     = {p_mw:.4f}')
print(f'  Decision (α=0.05): {"Reject H0" if p_mw < 0.05 else "Fail to reject H0"}')

print('\n=== 7.4 Effect Size (Cohen\'s d) ===')
pooled_std = np.sqrt((cis_growth_rates.std()**2 + non_cis_growth_rates.std()**2) / 2)
cohens_d   = (non_cis_growth_rates.mean() - cis_growth_rates.mean()) / pooled_std
print(f'  Cohen\'s d = {cohens_d:.4f}  ({"small" if abs(cohens_d)<0.5 else "medium" if abs(cohens_d)<0.8 else "large"} effect)')

## 8. Literacy Analysis

In [ ]:
print('=== 8.1 Literacy Data — All Categories ===')
display(literacy_clean.set_index('Category'))

print('\n=== 8.2 Percentage Change Between Census Years ===')
for col_a, col_b in [('1989','1999'),('1999','2009'),('2009','2022')]:
    df = literacy_clean.copy()
    df[f'Change {col_a}-{col_b}%'] = ((df[col_b] - df[col_a]) / df[col_a] * 100).round(1)
    print(f'\n  {col_a} → {col_b}:')
    display(df[['Category', col_a, col_b, f'Change {col_a}-{col_b}%']])

print('\n=== 8.3 Key Figures ===')
higher = literacy_clean[literacy_clean['Category'].str.contains('Higher professional', na=False)].iloc[0]
illit  = literacy_clean[literacy_clean['Category'].str.lower().str.strip()=='illiterates'].iloc[0]
print(f'  Higher education 1989→2022: {higher["1989"]:,.0f} → {higher["2022"]:,.0f}  (+{(higher["2022"]/higher["1989"]-1)*100:.1f}%)')
print(f'  Illiterates      1989→2022: {illit["1989"]:,.0f} → {illit["2022"]:,.0f}  ({(illit["2022"]/illit["1989"]-1)*100:+.1f}%)')

## 9. Summary Table

In [ ]:
summary = pd.DataFrame([
    ['CIS students 2010',               f'{students_df["CIS"].iloc[0]:,.0f}'],
    ['CIS students 2024',               f'{students_df["CIS"].iloc[-1]:,.0f}'],
    ['CIS CAGR 2010-2024',              f'{((students_df["CIS"].iloc[-1]/students_df["CIS"].iloc[0])**(1/14)-1)*100:.2f}%'],
    ['Non-CIS students 2010',           f'{students_df["Non_CIS"].iloc[0]:,.0f}'],
    ['Non-CIS students 2024',           f'{students_df["Non_CIS"].iloc[-1]:,.0f}'],
    ['Non-CIS CAGR 2010-2024',          f'{((students_df["Non_CIS"].iloc[-1]/students_df["Non_CIS"].iloc[0])**(1/14)-1)*100:.2f}%'],
    ['Peak total enrollment (year)',    f'{students_df["Total"].max():,.0f} ({students_df.loc[students_df["Total"].idxmax(),"Year"]})'],
    ['Pearson r (Total vs Budget)',     f'{stats.pearsonr(merged["Total"], merged["Total_mln_som"])[0]:.4f}'],
    ['Regression R²',                   f'{linregress(merged["Total_mln_som"], merged["Total"]).rvalue**2:.4f}'],
    ['Welch t-test p (one-tailed)',     f'{stats.ttest_ind(non_cis_growth_rates, cis_growth_rates, equal_var=False)[1]/2:.4f}'],
    ['Budget 2001 (mln som)',           f'{budget_ts["Total_mln_som"].iloc[0]:,.0f}'],
    ['Budget 2024 (mln som)',           f'{budget_ts["Total_mln_som"].iloc[-1]:,.0f}'],
    ['Budget CAGR 2001-2024',           f'{((budget_ts["Total_mln_som"].iloc[-1]/budget_ts["Total_mln_som"].iloc[0])**(1/23)-1)*100:.2f}%'],
], columns=['Metric','Value'])
display(summary)